In [ ]:
!pip install git+https://github.com/huggingface/transformers
# !pip install librosa
!pip install evaluate>=0.30
!pip install jiwer
# !pip install gradio
!pip install -q bitsandbytes datasets accelerate
!pip install git+https://github.com/huggingface/peft.git@main

In [ ]:
20°58'26.1"N 105°52'09.7"E

In [ ]:
from modules.core import llm_chain

llm_chain.start_ollama_server()

print("Gõ 'exit' hoặc 'quit' để thoát.")
print("Để gửi ảnh: image <đường_dẫn_ảnh> [nội dung_tin_nhắn]")

while True:
    user_input = input("\nYou: ").strip()
    if user_input.lower() in ["exit", "quit"]:
        print("🛑 Kết thúc chat.")
        break

    image_path = None
    text = user_input

    if user_input.lower().startswith("image "):
        parts = user_input.split(" ", 2)
        if len(parts) > 1:
            image_path = parts[1]
            text = parts[2] if len(parts) > 2 else ""
        else:
            print("⚠️ Cú pháp sai. Dùng: image <đường_dẫn_ảnh> [tin nhắn]")
            continue

    bot_response = llm_chain.chat(session_id=12, message=text, image_path=image_path)
    print(f"\nAssistant: {bot_response}")

In [ ]:
import os
import modules.config as config
os.path.exists(config.MEMORY_CHAT_PATH)

In [ ]:
import ollama

messages = [{"role": "system", "content": """Bạn là trợ lý AI /no_think"""},
            {"role": "user", "content": """1+1=?, so sánh 11.91 và 11.19 """}]
stream = ollama.chat(model="qwen3:8b", messages=messages,
                        stream=True)

In [ ]:
for stre in stream:
    print(stre.message.content, end="")

In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, WhisperProcessor
from transformers import pipeline
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

addapter_model = "TEST/checkpoint-116160"

peft_config = PeftConfig.from_pretrained(addapter_model)

# 2. Tải model Whisper gốc
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    peft_config.base_model_name_or_path,
    # "suzii/vi-whisper-large-v3-turbo-v1",
    torch_dtype=torch.float16,
    device_map="cuda"
)
model.generation_config.language = "<|vi|>"
model.generation_config.task = "transcribe"
model.config.forced_decoder_ids = None
# 3. Gắn adapter LoRA vào model
model = PeftModel.from_pretrained(model, addapter_model)
processor = WhisperProcessor.from_pretrained(
    peft_config.base_model_name_or_path,
    # "suzii/vi-whisper-large-v3-turbo-v1",
    language="vi", task="transcribe")

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    # device="cuda",
)

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
from transformers import pipeline
import os
import warnings
warnings.filterwarnings("ignore")

# Tạo Flask app
app = Flask(__name__)

# Tạo route API
@app.route("/transcribe", methods=["POST"])
def transcribe():
    if "file" not in request.files:
        return jsonify({"error": "No file uploaded"}), 400

    file = request.files["file"]
    file_path = "temp_audio.wav"
    file.save(file_path)
    
    result = pipe(file_path, return_timestamps=True, generate_kwargs = {"task": "transcribe", "num_beams": 1, "language":"<|vi|>"})
    text = result["text"]
    os.remove(file_path)

    return jsonify({"transcription": text})

# Mở tunnel ngrok thủ công
public_url = ngrok.connect(5000)
print("🌐 Ngrok URL:", public_url)

# Chạy Flask server
app.run(port=5000)
# https://3c20-35-239-153-178.ngrok-free.app/

In [ ]:
import cv2
import numpy as np

img = cv2.imread('TEST/Screenshot 2025-08-28 152449.png')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

blurred = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blurred, 75, 200)


contours, _ = cv2.findContours(edges.copy(), cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
contours = sorted(contours, key=cv2.contourArea, reverse=True)

for contour in contours:
    peri = cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, 0.02 * peri, True)
    
    if len(approx) == 4:
        paper_contour = approx
        break

def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")

    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]  # top-left
    rect[2] = pts[np.argmax(s)]  # bottom-right

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]  # top-right
    rect[3] = pts[np.argmax(diff)]  # bottom-left

    return rect

# Lấy góc và biến đổi ảnh
pts = paper_contour.reshape(4, 2)
rect = order_points(pts)
(tl, tr, br, bl) = rect

widthA = np.linalg.norm(br - bl)
widthB = np.linalg.norm(tr - tl)
maxWidth = max(int(widthA), int(widthB))

heightA = np.linalg.norm(tr - br)
heightB = np.linalg.norm(tl - bl)
maxHeight = max(int(heightA), int(heightB))

dst = np.array([
    [0, 0],
    [maxWidth - 1, 0],
    [maxWidth - 1, maxHeight - 1],
    [0, maxHeight - 1]], dtype="float32")

M = cv2.getPerspectiveTransform(rect, dst)
warped = cv2.warpPerspective(img, M, (maxWidth, maxHeight))

cv2.imshow("Scanned", warped)
cv2.waitKey(0)



In [ ]:
# docker run -p 8070:8070 -v C:\Users\ADMIN\Downloads\blackbox\weights:/models --gpus all ghcr.io/ggml-org/llama.cpp:server-cuda -m models/Qwen3VL-8B-Instruct-Q4_K_M.gguf -c 5000 --host 0.0.0.0 --port 8070 --n-gpu-layers 36 --jinja --mmproj models/mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf --parallel 1 -b 4096 -ub 4096



In [ ]:
import os
import pathlib
audio = pathlib.Path("TEST\output_segments\20251016_110347\SPEAKER_00_0p000_4p925s.wav")

In [ ]:
type(audio)

In [ ]:

def speech_diarization(audio_file, merge_gap_threshold=5, min_segment_duration=0.5, num_speaker=2):
    """
    Minimal placeholder for speech diarization.

    This function will ensure an output directory exists, copy the input audio into it (or
    convert to WAV if necessary), and return a message, a list of available WAV files in the
    output directory for the dropdown, and the output directory path (to be stored in state).

    Args mirror the call in `main.py`.
    """
    if not audio_file:
        return "❌ Không có file đầu vào", [], None

    if not os.path.exists(audio_file):
        return f"❌ File không tồn tại: {audio_file}", [], None

    out_dir = _ensure_output_dir()

    # If pyannote is available and a HF token is present, run diarization
    hf_token = os.environ.get('HUGGINGFACE_TOKEN')
    # print(hf_token)

    if _PYANNOTE_AVAILABLE and hf_token:
        try:
            pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-community-1', token=hf_token)
            diarization = pipeline(audio_file)

            # load audio with librosa (mono)
            data, sr = librosa.load(audio_file, sr=None, mono=True)

            saved_files = []
            diarization_lines = []
            seg_count = 0
            for turn, _, speaker in diarization.itertracks(yield_label=True):
                start = float(turn.start)
                end = float(turn.end)
                dur = end - start
                if dur < min_segment_duration:
                    continue

                s_idx = int(max(0, round(start * sr)))
                e_idx = int(min(len(data), round(end * sr)))
                segment = data[s_idx:e_idx]

                speaker_name = str(speaker).replace(' ', '_')
                seg_fname = f"{os.path.splitext(os.path.basename(audio_file))[0]}_{speaker_name}_{seg_count}.wav"
                seg_path = os.path.join(out_dir, seg_fname)
                sf.write(seg_path, segment, sr)
                saved_files.append(seg_fname)

                diarization_lines.append(f"{speaker} {start:.2f} {end:.2f}")
                seg_count += 1

            # write diarization transcript
            transcript_path = os.path.join(out_dir, os.path.splitext(os.path.basename(audio_file))[0] + '_diarization.txt')
            with open(transcript_path, 'w', encoding='utf-8') as f:
                for line in diarization_lines:
                    f.write(line + '\n')

            wavs = [f for f in os.listdir(out_dir) if f.lower().endswith('.wav')]
            message = f"✅ Diarization complete. Saved {len(saved_files)} segments to {out_dir}"
            return message, wavs, out_dir
        except Exception as e:
            tb = traceback.format_exc()
            # Fall back to placeholder behavior but include error details in message
            message = f"❌ Diarization failed: {e}\n{tb}\nFalling back to simple copy behavior."
            # continue to fallback

    # Fallback: copy/convert input file into output folder and return list
    base = os.path.basename(audio_file)
    name, ext = os.path.splitext(base)
    dest = os.path.join(out_dir, base)

    try:
        if ext.lower() == '.wav':
            shutil.copy2(audio_file, dest)
        else:
            data, sr = sf.read(audio_file)
            dest = os.path.join(out_dir, f"{name}.wav")
            sf.write(dest, data, sr)
    except Exception as e:
        return f"❌ Lỗi khi lưu/convert file: {e}", [], None

    wavs = [f for f in os.listdir(out_dir) if f.lower().endswith('.wav')]
    message = f"✅ Diarization placeholder: saved {os.path.basename(dest)} to {out_dir} (no real diarization performed)."
    return message, wavs, out_dir


In [ ]:
print("Hi")

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

model_id = r"C:\Users\ADMIN\Downloads\blackbox\weights\voice_vi"

processor = SpeechT5Processor.from_pretrained(model_id)
model = SpeechT5ForTextToSpeech.from_pretrained(model_id)
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

In [ ]:
from ddgs import DDGS

results = DDGS().text("python programming", max_results=5)
print(results)

In [ ]:
results = DDGS().images(
    query="butterfly",
    region="us-en",
    safesearch="off",
    timelimit="m",
    page=1,
    backend="auto",
    size=None,
    color="Monochrome",
    type_image=None,
    layout=None,
    license_image=None,
)
print(results)

In [1]:
import modules.tools.tool_login as tool_login

tool_login.login_and_click()

2026-02-25 15:29:03,395 - INFO - ====== WebDriver manager ======
2026-02-25 15:29:04,179 - INFO - Get LATEST chromedriver version for google-chrome
2026-02-25 15:29:04,424 - INFO - Get LATEST chromedriver version for google-chrome
2026-02-25 15:29:04,591 - INFO - Driver [C:\Users\ADMIN\.wdm\drivers\chromedriver\win64\145.0.7632.117\chromedriver-win32/chromedriver.exe] found in cache


Đăng nhập thành công, URL: http://10.0.99.101:3012/
Chờ nút xuất hiện: Lịch sử
✓ Nút 'Lịch sử' đã xuất hiện
✓ Nút 'Lịch sử' đã hiển thị
✓ Nút 'Lịch sử' đã sẵn sàng click
✓ Đã click nút 'Lịch sử'

Mã nguồn của trang 'Lịch sử':

➜ Kiểm tra trạng thái chấm công cho ngày 25-02-2026...
✓ Đã chấm công vào lúc: 08:11 25-02-2026
➜ Đã chấm công vào rồi. Kết thúc quá trình.
Đã thao tác xong và lưu screenshot: downloads/cache/screenshot.png
Đã dọn dẹp thư mục tạm: C:\Users\ADMIN\AppData\Local\Temp\tmpvhudp77w


'downloads/cache/screenshot.png'

In [1]:
import zipfile

with zipfile.ZipFile(r"C:\Users\ADMIN\Downloads\Untitled design.pptx", "r") as zip_ref:
    zip_ref.extractall(r"C:\Users\ADMIN\Downloads\pptx_contents")

In [17]:
import re
import torch
from tqdm import tqdm
import pickle
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from scipy.signal import savgol_filter
import soundfile as sf


def split_into_sentences(text):
    sentences = re.split(r'[.!?]+\s+', text.strip())
    return [sentence.strip() for sentence in sentences if sentence.strip()]


def clone_tts(para, speaker_emb, model, processor, vocoder, device):
    spectrogram_audio = None

    model = model.to(device)
    vocoder = vocoder.to(device)
    speaker_emb = speaker_emb.to(device)

    for text in tqdm(split_into_sentences(para)):
        text = text.replace("\n", " ").strip()
        if len(text) == 0:
            continue

        inputs = processor(text=text.lower(), return_tensors="pt")
        input_ids = inputs["input_ids"].to(device)

        with torch.no_grad():
            spectrogram = model.generate_speech(input_ids, speaker_emb)

        if spectrogram_audio is None:
            spectrogram_audio = spectrogram
        else:
            spectrogram_audio = torch.cat((spectrogram_audio, spectrogram), dim=0)

    with torch.no_grad():
        speech = vocoder(spectrogram_audio)

    return speech.cpu().numpy()


def init_model(processor_id="dolphinnlp/voice_vi", model_id="dolphinnlp/voice_vi", vocoder_id="vocoder"):
    processor = SpeechT5Processor.from_pretrained(processor_id)
    model = SpeechT5ForTextToSpeech.from_pretrained(model_id)
    vocoder = SpeechT5HifiGan.from_pretrained(vocoder_id)
    return processor, model, vocoder


def run(output_filename, text, window_length=5, polyorder=1, sample_rate=24000):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    with open(r'C:\Users\ADMIN\Downloads\blackbox\storage\data\voice_speaker_vy.pkl', 'rb') as f:
        speaker_embeddings_loaded = pickle.load(f)

    speaker_embeddings = speaker_embeddings_loaded[80].unsqueeze(0)

    model_id = r"C:\Users\ADMIN\Downloads\blackbox\weights\voice_vi"

    processor, model, vocoder = init_model(
        processor_id=model_id,
        model_id=model_id,
        vocoder_id="microsoft/speecht5_hifigan"
    )

    speech = clone_tts(text, speaker_embeddings, model, processor, vocoder, device)

    smoothed_audio = savgol_filter(speech, window_length, polyorder)

    sf.write(output_filename, smoothed_audio, sample_rate)

    return output_filename




In [18]:
run(r"C:\Users\ADMIN\Downloads\blackbox\output.wav", """Ái bi ti Tương lai sáng tạo, nơi trí tuệ hội tụ
Mỗi dòng code là khát vọng, mỗi thuật toán là ước mơ,
Ái bi ti vươn mình trong thế giới số,
Tạo nên sản phẩm thông minh, phục vụ con người chân thành.

Từ phòng lab đến thị trường toàn cầu,
Từng ý tưởng đều được gìn giữ, từng ý tưởng đều được hiện thực.
Nhân viên trẻ, tâm huyết, sáng tạo không ngừng,
Làm nên kỳ tích từ tư duy đến thực tế.

Ái bi ti không chỉ phát triển công nghệ,
Mà còn nuôi dưỡng nhân tài, xây dựng văn hóa đổi mới.
Vì một tương lai thông minh, bền vững và nhân văn,
Ái bi ti dẫn đầu, vì cộng đồng, vì tương lai.""", window_length=5, polyorder=1, sample_rate=24000)

Device: cuda


100%|██████████| 5/5 [00:13<00:00,  2.67s/it]


'C:\\Users\\ADMIN\\Downloads\\blackbox\\output.wav'